# A. FFT separation – music + speech (v1 baseline, FFT only)

**Environment: Colab only.** Runs server-side, no Drive mount needed, no large downloads.
Local counterpart: `../local/baseline_fft_separation.py` (same method, no GPU, tiny local clips only).

Structure: Setup -> synthetic mix -> STFT -> band separation -> ISTFT -> plots. No training, no torch, no data cost to you beyond opening the notebook. Real clips come later as a small slice.


In [ ]:
# Cell 1 - Setup (safe for Colab, no heavy downloads)
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
SR = 44100
NFFT, NOVERLAP = 2048, 1536
print('Setup ok, SR =', SR)

In [ ]:
# Cell 2 - Synthetic mix of two tones (demo stands in for music + speech)
t = np.arange(int(SR * 4.0)) / SR
s1 = 0.6*np.sin(2*np.pi*220*t) + 0.2*np.sin(2*np.pi*440*t)
s2 = 0.5*np.sin(2*np.pi*330*t) + 0.2*np.sin(2*np.pi*660*t)
mix = (s1 + s2) * 0.5
plt.figure(figsize=(10,3)); plt.plot(t[::10], mix[::10]); plt.title('Mixed signal (downsampled for view)'); plt.show()

In [ ]:
# Cell 3 - STFT + band separation + ISTFT
f, tt, Z = signal.stft(mix, fs=SR, nperseg=NFFT, noverlap=NOVERLAP)
mask1 = ((f>=150)&(f<280)).astype(float)[:,None]
mask2 = ((f>=260)&(f<380)).astype(float)[:,None]
Z1, Z2 = Z*mask1, Z*mask2
_, x1 = signal.istft(Z1, fs=SR, nperseg=NFFT, noverlap=NOVERLAP)
_, x2 = signal.istft(Z2, fs=SR, nperseg=NFFT, noverlap=NOVERLAP)
print('Recovered lengths:', len(x1), len(x2))

In [ ]:
# Cell 4 - Spectrogram view (v1 deliverable)
fig, ax = plt.subplots(1,3, figsize=(14,4))
for a, Z_, ttl in zip(ax, [Z,Z1,Z2], ['Mix','Band 220Hz','Band 330Hz']):
    a.pcolormesh(tt, f, 20*np.log10(np.abs(Z_)+1e-9), shading='gouraud', cmap='magma', vmin=-60, vmax=0)
    a.set_ylim(0,4000); a.set_title(ttl); a.set_xlabel('Time (s)'); a.set_ylabel('Hz')
plt.tight_layout(); plt.savefig('/tmp/music_speech_spectrograms.png', dpi=130); plt.show()
print('Saved to /tmp/music_speech_spectrograms.png')

## Next (not in this cell)
Replace `s1, s2, mix` with a tiny real slice (MUSDB or 5 local WAVs < 50 MB) fetched in Colab. Keep the same STFT/band code. Train nothing yet – FFT baseline is job one.